# Web and social media analysis
A project of social media analysis about these questions:

1. **Sentiment & framing** — Do people talk about MT as a tool, a threat, or both? How does framing differ between users (translators vs. general public vs. tech advocates)?
2. **Professional impact discourse** — How do working translators and localizers discuss MT's effect on their careers, rates, and job availability?
3. **Utility vs. displacement tension** — Do people acknowledge MT's utility while also lamenting professional erosion, or are these largely separate conversations?

## Setup

In [1]:
!pip -q install pandas atproto python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.2/553.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.8/285.8 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import time
import re
from datetime import datetime, timezone, timedelta
import pandas as pd

from atproto import Client, models
from atproto_client.exceptions import RequestException, InvokeTimeoutError, NetworkError
from dotenv import load_dotenv
import os

## Data gathering

Use BlueSky social media, their API to fetch posts, threads, users and crawl to get connections.

In [3]:
# SearchPostsResponse
# │
# ├── posts: List[PostView]
# │   │
# │   └── PostView
# │       ├── uri          # unique ID of the post (e.g. at://did:plc:.../post/abc123)
# │       ├── cid          # content hash
# │       ├── like_count
# │       ├── repost_count
# │       ├── reply_count
# │       │
# │       ├── author: ProfileViewBasic
# │       │   ├── handle       # e.g. "janedoe.bsky.social"
# │       │   ├── display_name
# │       │   └── description  # ← this is the bio, useful for user labeling
# │       │
# │       └── record: Record   # ← the actual post content
# │           ├── text         # ← the post text you want
# │           ├── created_at   # timestamp
# │           └── reply: Optional
# │               ├── parent.uri   # uri of the post this replies to
# │               └── root.uri     # uri of the thread root
# │
# └── cursor   # pagination token — pass this to get the next page of results

Data gathering pipeline:

1. Search keywords → seed posts
2. Crawl threads of each seed post → more posts + reply structure (needed for graph!)
3. Crawl following of top authors → more users
4. Deduplicate everything → final dataset

In [10]:
from google.colab import userdata
os.environ["BSKY_HANDLE"] = userdata.get("BSKY_HANDLE")
os.environ["BSKY_PASSWORD"] = userdata.get("BSKY_PASSWORD")

In [11]:
print(repr(os.getenv("BSKY_HANDLE")))
print(repr(os.getenv("BSKY_PASSWORD")))

'su-p04.bsky.social'
'Lj5z2E6Y5Si7cLX'


In [12]:
client = Client()
client.login(os.getenv("BSKY_HANDLE"), os.getenv("BSKY_PASSWORD"))

ProfileViewDetailed(did='did:plc:qlj5cxjmj4kepbbbiektn4b2', handle='su-p04.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, germ=None, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:qlj5cxjmj4kepbbbiektn4b2/bafkreiaus5ezpmcwtoe2iomo4dpazxnoe2bdf2gdkdohgw42l3hd5nggiu', banner=None, created_at='2026-06-26T13:48:37.562Z', debug=None, description=None, display_name='', followers_count=0, follows_count=1, indexed_at='2026-06-26T13:49:03.355Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=0, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None,

In [13]:
def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)
        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")

                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)

                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(10, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")

In [14]:
# Facet extraction: hashtags, mentions, links

def extract_facets(record) -> dict:
    """Extract hashtags, mentions, links from a post record (if facets exist)."""
    hashtags = []
    mentions = []
    links = []

    facets = getattr(record, "facets", None) or []
    for facet in facets:
        features = getattr(facet, "features", None) or []
        for feat in features:
            ftype = getattr(feat, "py_type", None) or getattr(feat, "type", None)

            if ftype == "app.bsky.richtext.facet#tag":
                tag = getattr(feat, "tag", None)
                if tag:
                    hashtags.append(tag)
            elif ftype == "app.bsky.richtext.facet#mention":
                did = getattr(feat, "did", None)
                if did:
                    mentions.append({"did": did})
            elif ftype == "app.bsky.richtext.facet#link":
                uri = getattr(feat, "uri", None)
                if uri:
                    links.append(uri)

    return {"hashtags": hashtags, "mentions": mentions, "links": links}

In [15]:
# Search function with time window and facets extraction
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def search_posts_time_window(
    client,
    query: str = None,
    tag: str = None,
    since_iso: str = None,
    until_iso: str = None,
    max_posts: int = 1000,
    page_size: int = 50,
    polite_sleep: float = 0.25,
    print_every_page: bool = False,
):
    cursor = None
    rows = []
    page = 0

    while len(rows) < max_posts:
        page += 1

        params = models.AppBskyFeedSearchPosts.Params(
            q=query if query else tag,
            tag=[tag] if tag else None,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, max_posts - len(rows)),
            cursor=cursor,
        )

        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            if print_every_page:
                print(f"[page {page}] no posts, stopping.")
            break

        if print_every_page:
            dts = []
            for p in posts:
                created = getattr(getattr(p, 'record', None), 'created_at', None)
                if created:
                    dts.append(_parse_dt_utc(created))
            if dts:
                print(
                    f"[page {page}] newest={max(dts).isoformat()}  oldest={min(dts).isoformat()}  "
                    f"collected={len(rows)}  cursor={'yes' if cursor else 'no'}"
                )

        for p in posts:
            rec = getattr(p, "record", None)
            created = getattr(rec, "created_at", None)
            if not created:
                continue

            created_dt = _parse_dt_utc(created).strftime("%Y-%m-%dT%H:%M:%S.000Z")
            if not (since_iso <= created_dt < until_iso):
                continue

            author = getattr(p, "author", None)
            facets_data = extract_facets(rec)
            reply = getattr(rec, "reply", None)

            rows.append({
                "created_at": created_dt,
                "uri": p.uri,
                "cid": getattr(p, "cid", None),
                "text": getattr(rec, "text", None),
                "handle": getattr(author, "handle", None),
                "did": getattr(author, "did", None),
                "replies": getattr(p, "reply_count", None),
                "likes": getattr(p, "like_count", None),
                "reposts": getattr(p, "repost_count", None),
                "quotes": getattr(p, "quote_count", None),
                "hashtags": facets_data["hashtags"],
                "mentions": facets_data["mentions"],
                "links": facets_data["links"],
                "is_reply": reply is not None,
                "reply_to": reply.parent.uri if reply else None,
                "query": query or f"#{tag}",
            })

            if len(rows) >= max_posts:
                break

        if cursor is None:
            if print_every_page:
                print(f"[page {page}] cursor is None, stopping.")
            break

        time.sleep(polite_sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
        df = df.sort_values("created_at", ascending=True).reset_index(drop=True)
    return df

In [16]:
def get_followers_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollowers.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_followers, params)
        cursor = res.cursor
        followers = res.followers or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in followers])

        # print(f"[followers page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not followers:
            break
        time.sleep(polite_sleep)
    return out


def get_following_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollows.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_follows, params)
        cursor = res.cursor
        follows = res.follows or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in follows])

        # print(f"[following page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not follows:
            break
        time.sleep(polite_sleep)
    return out

In [17]:
# Time window: last 12 months
UNTIL_ISO = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
SINCE_ISO = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Collecting from {SINCE_ISO} to {UNTIL_ISO}")

### Seed Crawl

In [18]:
# Define queries and hastags to start from
TEXT_QUERIES = [
    '"machine translation"',
    '"AI translation"',
    '"post-editing"',
    '"human translation"',
    '"translation quality"',
    '"translator jobs"',
    '"translation industry"',
    'DeepL',
    'MTPE',
    'neural machine translation',
]

COMPOUND_QUERIES = [
    '"post-editing" AI',
    '"AI translation" threat',
    'scraped translation',
    '"machine translation" slop',
    '"human translator" AI',
    '"translation industry" AI',
]

HASHTAG_QUERIES = [
    'machinetranslation',
    'MTPE',
    'localization',
    'l10n',
    'SlowTranslation',
    'HumanTranslation',
    'PauseAI',
    'NoAI',
]

In [19]:
# API data fetching

all_dfs = []

for q in TEXT_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=300, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

for tag in HASHTAG_QUERIES:
    df_t = search_posts_time_window(
        client, tag=tag,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=300, page_size=25, polite_sleep=0.35,
    )
    df_t['query'] = f'#{tag}'
    all_dfs.append(df_t)

for q in COMPOUND_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=300, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

df_seeds = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='uri').reset_index(drop=True)
print(f"\nTotal seed posts: {len(df_seeds)}")
print(f"Unique authors: {df_seeds['handle'].nunique()}")
print(f"\nBreakdown:\n{df_seeds['query'].value_counts()}")


Total seed posts: 3765
Unique authors: 2420

Breakdown:
query
"machine translation"         300
#NoAI                         300
#localization                 299
#l10n                         298
"AI translation"              297
MTPE                          296
DeepL                         293
"post-editing"                292
"human translation"           292
"translation quality"         282
#PauseAI                      263
"translation industry"        145
neural machine translation    106
"human translator" AI          87
"post-editing" AI              74
#machinetranslation            69
"machine translation" slop     23
scraped translation            21
"translator jobs"              19
#HumanTranslation               6
"AI translation" threat         3
Name: count, dtype: int64


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
os.makedirs('/content/drive/MyDrive/bluesky-data', exist_ok=True)

In [25]:
print(df_seeds.columns)
df_seeds.to_csv('/content/drive/MyDrive/bluesky-data/seeds.csv', index=False)
print("Saved seeds.csv")

Index(['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes',
       'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply',
       'reply_to', 'query'],
      dtype='object')
Saved seeds.csv


In [26]:
# Crawl the threads of seed posts

visited_uris = set(df_seeds['uri'].tolist())  # skip re-fetching the seeds
thread_posts = []
reply_edges = []

def crawl_thread(uri, depth=0, max_depth=4):
    if depth > max_depth:
        return

    try:
        thread = client.app.bsky.feed.get_post_thread({'uri': uri, 'depth': 1})
        node = thread.thread

        if not hasattr(node, 'post'):
            return

        post = node.post

        # collect this post only if not already seen
        if post.uri not in visited_uris:
            visited_uris.add(post.uri)
            facets = extract_facets(post.record)
            thread_posts.append({
                'uri':        post.uri,
                'text':       post.record.text,
                'created_at': post.record.created_at,
                'handle':     post.author.handle,
                'likes':      post.like_count,
                'reposts':    post.repost_count,
                'replies':    post.reply_count,
                'is_reply':   post.record.reply is not None,
                'reply_to':   post.record.reply.parent.uri if post.record.reply else None,
                'hashtags':   facets['hashtags'],
                'mentions':   facets['mentions'],
                'links':      facets['links'],
                'query':      'thread_crawl',
            })

            if post.record.reply:
                reply_edges.append({
                    'source': post.author.handle,
                    'target_uri': post.record.reply.parent.uri,
                    'type': 'reply',
                })

        # always recurse into replies regardless
        if hasattr(node, 'replies') and node.replies:
            for reply in node.replies:
                if hasattr(reply, 'post'):
                    crawl_thread(reply.post.uri, depth + 1, max_depth)

        time.sleep(0.3)

    except Exception as e:
        print(f"  skipped {uri}: {e}")

# only crawl seeds that have replies — no point crawling dead threads
seeds_with_replies = df_seeds[df_seeds['replies'] > 0]['uri'].tolist()
print(f"Seeds with replies: {len(seeds_with_replies)} / {len(df_seeds)}")

for i, uri in enumerate(seeds_with_replies):
    if i % 20 == 0:
        print(f"  crawling {i}/{len(seeds_with_replies)}...")
    crawl_thread(uri)

df_threads = pd.DataFrame(thread_posts).drop_duplicates(subset='uri').reset_index(drop=True)
df_reply_edges = pd.DataFrame(reply_edges)

print(f"\nNew posts from threads: {len(df_threads)}")
print(f"Reply edges found: {len(df_reply_edges)}")

Seeds with replies: 1496 / 3765
  crawling 0/1496...
  crawling 20/1496...
  crawling 40/1496...
  crawling 60/1496...
  crawling 80/1496...
  crawling 100/1496...
  crawling 120/1496...
  crawling 140/1496...
  crawling 160/1496...
  crawling 180/1496...
  crawling 200/1496...
  crawling 220/1496...
  crawling 240/1496...
  crawling 260/1496...
  crawling 280/1496...
  crawling 300/1496...
  crawling 320/1496...
  crawling 340/1496...
  crawling 360/1496...
  crawling 380/1496...
  crawling 400/1496...
  crawling 420/1496...
  crawling 440/1496...
  crawling 460/1496...
  crawling 480/1496...
  crawling 500/1496...
  crawling 520/1496...
  crawling 540/1496...
  crawling 560/1496...
  crawling 580/1496...
  crawling 600/1496...
  crawling 620/1496...
  crawling 640/1496...
  crawling 660/1496...
  crawling 680/1496...
  crawling 700/1496...
  crawling 720/1496...
  crawling 740/1496...
  crawling 760/1496...
  crawling 780/1496...
  crawling 800/1496...
  crawling 820/1496...
  crawli

In [27]:
df_seeds.head()

,created_at,uri,cid,text,handle,did,replies,likes,reposts,quotes,hashtags,mentions,links,is_reply,reply_to,query
0,2026-06-09 14:53:20+00:00,at://did:plc:7soujtbrvjfe4zbdc6yujra4/app.bsky...,bafyreifmgzk62g5golzocvs5n7t4ngjamphfhqeq2ld2l...,Context is exactly the thing a context-blind p...,loekalization.bsky.social,did:plc:7soujtbrvjfe4zbdc6yujra4,0.0,0.0,0.0,0.0,[],[],[],True,at://did:plc:7soujtbrvjfe4zbdc6yujra4/app.bsky...,"""machine translation"""
1,2026-06-09 15:02:38+00:00,at://did:plc:7soujtbrvjfe4zbdc6yujra4/app.bsky...,bafyreigangr3a7htcfiu4d6ixbvwtzfc663yphoqwx6lm...,It is choosing the right meaning for this obje...,loekalization.bsky.social,did:plc:7soujtbrvjfe4zbdc6yujra4,0.0,1.0,0.0,0.0,[],[],[],True,at://did:plc:7soujtbrvjfe4zbdc6yujra4/app.bsky...,"""machine translation"""
2,2026-06-09 16:28:13+00:00,at://did:plc:uw6jvomwiq3cowtelzyu5vtm/app.bsky...,bafyreicsewtmam5xu4vhs5fe2jwaywgt72cnjhuvsvgfr...,You could say it's a parallel situation. Basic...,cossackgene.bsky.social,did:plc:uw6jvomwiq3cowtelzyu5vtm,0.0,3.0,0.0,0.0,[],[],[],True,at://did:plc:ymvwq7hwhepeeeyu5nbpgd6m/app.bsky...,"""machine translation"""
3,2026-06-09 16:41:28+00:00,at://did:plc:hip3cw3dlkoljfotexuloe63/app.bsky...,bafyreigvtpvith74gmmxml4tyfhxlj3cdzuiex6ao3vub...,The rocket slime GBA game got translated but i...,fm-synth.bsky.social,did:plc:hip3cw3dlkoljfotexuloe63,1.0,0.0,0.0,0.0,[],[],[],True,at://did:plc:pqexvwl6pm6jqjnqmuphkdqw/app.bsky...,"""machine translation"""
4,2026-06-09 16:43:19+00:00,at://did:plc:z4qetjanxuehao63dbrbytfm/app.bsky...,bafyreibk2isjrk6m7yag3txsz5yjrunfyt2ppkl6mqrxp...,"Alors en fait le ""Koji Shimada"" c'est une erre...",ninjabird.bsky.social,did:plc:z4qetjanxuehao63dbrbytfm,1.0,1.0,0.0,0.0,[],[],[],True,at://did:plc:twxtyto2nreoo5mlclhjbjhz/app.bsky...,"""machine translation"""


In [28]:
df_all_posts = pd.concat([df_seeds, df_threads], ignore_index=True)
df_all_posts = df_all_posts.drop_duplicates(subset='uri').reset_index(drop=True)

print(f"Total posts: {len(df_all_posts)}")
print(f"Unique authors: {df_all_posts['handle'].nunique()}")

df_all_posts.to_csv('/content/drive/MyDrive/bluesky-data/posts_raw.csv', index=False)
df_reply_edges.to_csv('/content/drive/MyDrive/bluesky-data/reply_edges.csv', index=False)
print("Saved posts_raw.csv and reply_edges.csv")

Total posts: 8244
Unique authors: 4075
Saved posts_raw.csv and reply_edges.csv


In [29]:
# Only crawl the followings of the most active users, not all
author_counts = df_all_posts['handle'].value_counts()
top_authors = author_counts[author_counts > 1].index.tolist()
print(f"Authors with more than 1 post: {len(top_authors)}")

Authors with more than 2 post: 1460


### Author / Follower Crawl

In [30]:
# ── AUTHOR/FOLLOWER CRAWL ──────────────────────────────
# Crawl the following of the top authors

follow_edges = []
author_profiles = []

for i, handle in enumerate(top_authors):

    try:
        # get bio
        profile = client.app.bsky.actor.get_profile({'actor': handle})
        author_profiles.append({
            'handle':       handle,
            'display_name': profile.display_name,
            'bio':          profile.description,
            'followers':    profile.followers_count,
            'following':    profile.follows_count,
            'posts_count':  profile.posts_count,
        })
        time.sleep(0.3)

        # get followers (capped at 100 per author)
        followers = get_followers_list(client, handle, max_total=100, page_size=100, polite_sleep=0.3)
        for f in followers:
            follow_edges.append({
                'source': f['handle'],
                'target': handle,
                'type':   'follows',
            })
        time.sleep(0.3)
        if i % 100 == 0:
            print(f"  crawling {i}/{len(top_authors)}...")
    except Exception as e:
        print(f"  skipped {handle}: {e}")

df_profiles = pd.DataFrame(author_profiles)
df_follow_edges = pd.DataFrame(follow_edges)

print(f"\nProfiles collected: {len(df_profiles)}")
print(f"Follow edges collected: {len(df_follow_edges)}")

df_profiles.to_csv('/content/drive/MyDrive/bluesky-data/author_profiles.csv', index=False)
df_follow_edges.to_csv('/content/drive/MyDrive/bluesky-data/follow_edges.csv', index=False)
print("Saved author_profiles.csv and follow_edges.csv")

  crawling 0/1460...
  crawling 100/1460...
  crawling 200/1460...
  crawling 300/1460...
  crawling 400/1460...
  crawling 500/1460...
  crawling 600/1460...
  crawling 700/1460...
  crawling 800/1460...
  crawling 900/1460...
  crawling 1000/1460...
  crawling 1100/1460...
  crawling 1200/1460...
  crawling 1300/1460...
  crawling 1400/1460...

Profiles collected: 1460
Follow edges collected: 118701
Saved author_profiles.csv and follow_edges.csv


In [31]:
print(df_profiles.head())
print(df_profiles['handle'].value_counts().head())

                     handle                            display_name  \
0        ohaasa.thekipa.com                                  오하아사 봇   
1  tamslator.eurosky.social  Tam | She/her #TranslatorsInTheCredits   
2    modilingua.bsky.social                              Modilingua   
3       pauseai.bsky.social                                 PauseAI   
4               liabelle.me                             Lia Belle 🌷   

                                                 bio  followers  following  \
0  おはよう朝日です(https://www.asahi.co.jp/ohaasa/ )의 별자...       2019          1   
1  (UTC+1)\nMaker of stories, wizard of jokes, an...       1080        358   
2  Localization, Multilingual SEO, LangOps, Conte...         33         13   
3  Community of volunteers who work together to m...        850         35   
4  Blogger. Content Marketer. News Junkie. Avid R...        495         37   

   posts_count  
0         9430  
1         2382  
2          319  
3          355  
4         1259  
ha

### Combine Edges


In [32]:
# ── COMBINE EDGES ────────────────────────────────────────────────

# reply edges need target handle, not uri — resolve later
# for now just label them differently and combine
df_reply_edges['source'] = df_reply_edges['source']
df_reply_edges['target'] = df_reply_edges['target_uri']  # uri for now
df_reply_edges = df_reply_edges[['source', 'target', 'type']]

df_graph = pd.concat([df_follow_edges, df_reply_edges], ignore_index=True)

print(f"Total graph edges: {len(df_graph)}")
print(f"  follow edges: {len(df_follow_edges)}")
print(f"  reply edges:  {len(df_reply_edges)}")

df_graph.to_csv('/content/drive/MyDrive/bluesky-data/graph.csv', index=False)
print("Saved graph.csv")

Total graph edges: 123180
  follow edges: 118701
  reply edges:  4479
Saved graph.csv


## Preprocessing
Deduplicating data, cleaning, removing non-english posts or posts that are not connected to the chosen topic.

In [33]:
# RUN FROM HERE ON SAVED RAW DATA - not calling API anymore, now local cleaning
df = pd.read_csv('/content/drive/MyDrive/bluesky-data/posts_raw.csv')
df_profiles = pd.read_csv('/content/drive/MyDrive/bluesky-data/author_profiles.csv')
df_reply_edges = pd.read_csv('/content/drive/MyDrive/bluesky-data/reply_edges.csv')
df_follow_edges = pd.read_csv('/content/drive/MyDrive/bluesky-data/follow_edges.csv')

print(f"Posts loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

Posts loaded: 8244
Columns: ['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes', 'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply', 'reply_to', 'query']


In [34]:
before = len(df)
df = df.drop_duplicates(subset='uri').reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicates, {len(df)} remaining")

Dropped 0 duplicates, 8244 remaining


In [35]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)       # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)

In [36]:
df = df[df['text_clean'].str.len() > 10].reset_index(drop=True)
print(f"Posts after empty drop: {len(df)}")

Posts after empty drop: 7942


In [37]:
# Topical relevance filtering
off_topic_queries = ['#PauseAI', '#NoAI', '#localization']

for q in off_topic_queries:
    print(f"\n── {q} ──")
    sample = df[df['query'] == q]['text_clean'].sample(min(5, len(df[df['query'] == q]))).tolist()
    for t in sample:
        print(f"  • {t[:120]}")


── #PauseAI ──
  • PauseAI oprichter Joep Meindertsma was afgelopen zaterdag in Nieuwsuur. Kijk hier het item: npo.nl/start/afspel... vanaf
  • The key message of PauseAI is not doom. It's hope.
  • AI is not just coming for your job. This PauseAI campaign has brought together hundreds of concerned citizens across the
  • “Right now, #AI companies are less #regulated than sandwich shops." —Ella Hughes, organizing director of activist group 
  • Moment of silence for pauseai

── #NoAI ──
  • youtube.com/shorts/aNTT2... The latest Roll for Art adventure is live! This time the theme is Technology! Come join the 
  • Bluesky's Top 10 Trending Words (past 10min): 💨x1* - iran 🔓 💨x1* - democracy 🔓 💨x1* - ukraine 🔓 💨x33 - haitians 💨x1* - e
  • Hey ladies and gentlemen tonight I'll be streaming on ttv at 3:30GMT. Would love to have you there... Come join in draw 
  • An Immortal's wish♾️ Alien worlds👽 Life reshaped🧬 A Queen's gift👑 Crusade against entropy⚔️ Eiepublishing.com books2read
  • Jud

In [38]:
MT_TERMS = [
    'translat', 'interpret', 'locali', 'localiz',  # root forms catch variants
    'deepl', 'mtpe', 'post-edit', 'post edit',
    'language model', 'nlp', 'multilingual',
    'linguist', 'terminology', 'glossary',
]

def is_relevant(text):
    text_lower = text.lower()
    return any(term in text_lower for term in MT_TERMS)

before = len(df)
df = df[df['text_clean'].apply(is_relevant)].reset_index(drop=True)
print(f"Dropped {before - len(df)} irrelevant posts")
print(f"Remaining: {len(df)}")
print(f"\nBreakdown by query:")
print(df['query'].value_counts())

Dropped 4175 irrelevant posts
Remaining: 3767

Breakdown by query:
query
thread_crawl                  798
"AI translation"              292
#localization                 290
MTPE                          287
DeepL                         285
"machine translation"         283
"post-editing"                282
"human translation"           280
"translation quality"         278
#l10n                         163
"translation industry"        143
neural machine translation    100
"human translator" AI          77
"post-editing" AI              73
#machinetranslation            69
"machine translation" slop     22
"translator jobs"              19
scraped translation            17
#HumanTranslation               6
"AI translation" threat         3
Name: count, dtype: int64


In [42]:
# User type labeling

PROFESSIONAL_TERMS = [
    'translat', 'interpret', 'locali', 'linguist',
    'terminolog', 'lexicograph', 'subtitl', 'transcrib'
]

TECH_TERMS = [
    'developer', 'engineer', 'nlp', 'machine learning', 'ai researcher',
    'data scientist', 'software', 'programmer', 'coder', 'researcher'
]

def label_user(bio):
    if not isinstance(bio, str):
        return 'general'
    bio = bio.lower()
    if any(t in bio for t in PROFESSIONAL_TERMS):
        return 'professional'
    if any(t in bio for t in TECH_TERMS):
        return 'tech'
    return 'general'

bio_map = dict(zip(df_profiles['handle'], df_profiles['bio']))
df['bio'] = df['handle'].map(bio_map)
df['user_type'] = df['bio'].apply(label_user)

print(df['user_type'].value_counts())

user_type
general         3118
professional     530
tech             119
Name: count, dtype: int64


In [43]:
df_profiles.head()

,handle,display_name,bio,followers,following,posts_count
0,ohaasa.thekipa.com,오하아사 봇,おはよう朝日です(https://www.asahi.co.jp/ohaasa/ )의 별자...,2019,1,9430
1,tamslator.eurosky.social,Tam | She/her #TranslatorsInTheCredits,"(UTC+1)\nMaker of stories, wizard of jokes, an...",1080,358,2382
2,modilingua.bsky.social,Modilingua,"Localization, Multilingual SEO, LangOps, Conte...",33,13,319
3,pauseai.bsky.social,PauseAI,Community of volunteers who work together to m...,850,35,355
4,liabelle.me,Lia Belle 🌷,Blogger. Content Marketer. News Junkie. Avid R...,495,37,1259


In [46]:
# resolve reply edges uri → handle
uri_to_handle = dict(zip(df['uri'], df['handle']))
df_reply_edges['target'] = df_reply_edges['target'].map(uri_to_handle)
df_reply_edges = df_reply_edges.dropna(subset=['target'])  # drop unresolvable
df_reply_edges = df_reply_edges[['source', 'target', 'type']]

# rebuild graph.csv with resolved handles
df_graph = pd.concat([df_follow_edges, df_reply_edges], ignore_index=True)
print(f"Reply edges resolved: {len(df_reply_edges)}")
print(f"Total graph edges: {len(df_graph)}")

# save clean datasets
df_graph.to_csv('/content/drive/MyDrive/bluesky-data/graph.csv', index=False)

print("\nSaved:")
print(f"  graph.csv        — {len(df_graph)} edges")

Reply edges resolved: 0
Total graph edges: 118701

Saved:
  graph.csv        — 118701 edges


In [47]:
# ── CONSISTENCY CHECKS ───────────────────────────────────────────

print("═" * 50)
print("POSTS CLEAN")
print("═" * 50)
print(f"Total posts:        {len(df)}")
print(f"Unique URIs:        {df['uri'].nunique()}  (should match total)")
print(f"Unique authors:     {df['handle'].nunique()}")
print(f"Null text:          {df['text_clean'].isnull().sum()}  (should be 0)")
print(f"Null handle:        {df['handle'].isnull().sum()}  (should be 0)")
print(f"Null created_at:    {df['created_at'].isnull().sum()}  (should be 0)")
print(f"Date range:         {df['created_at'].min()} → {df['created_at'].max()}")
print(f"User types:         {df['user_type'].value_counts().to_dict()}")

print("\n" + "═" * 50)
print("GRAPH")
print("═" * 50)
print(f"Total edges:        {len(df_graph)}")
print(f"Edge types:         {df_graph['type'].value_counts().to_dict()}")
print(f"Null source:        {df_graph['source'].isnull().sum()}  (should be 0)")
print(f"Null target:        {df_graph['target'].isnull().sum()}  (should be 0)")
print(f"Unique sources:     {df_graph['source'].nunique()}")
print(f"Unique targets:     {df_graph['target'].nunique()}")

print("\n" + "═" * 50)
print("PROFILES")
print("═" * 50)
print(f"Total profiles:     {len(df_profiles)}")
print(f"Null bios:          {df_profiles['bio'].isnull().sum()}")
print(f"All in posts:       {df_profiles['handle'].isin(df['handle']).all()}  (should be True)")

print("\n" + "═" * 50)
print("CROSS-FILE")
print("═" * 50)
graph_handles = set(df_graph['source']).union(set(df_graph['target']))
post_handles = set(df['handle'])
print(f"Graph handles in posts:  {len(graph_handles & post_handles)} / {len(graph_handles)}")
print(f"Reply sources in posts:  {df_reply_edges['source'].isin(post_handles).sum()} / {len(df_reply_edges)}")

══════════════════════════════════════════════════
POSTS CLEAN
══════════════════════════════════════════════════
Total posts:        3767
Unique URIs:        3767  (should match total)
Unique authors:     2296
Null text:          0  (should be 0)
Null handle:        0  (should be 0)
Null created_at:    0  (should be 0)
Date range:         2025-06-27 06:01:08+00:00 → 2026-06-26T13:36:28.942Z
User types:         {'general': 3118, 'professional': 530, 'tech': 119}

══════════════════════════════════════════════════
GRAPH
══════════════════════════════════════════════════
Total edges:        118701
Edge types:         {'follows': 118701}
Null source:        0  (should be 0)
Null target:        0  (should be 0)
Unique sources:     79492
Unique targets:     1449

══════════════════════════════════════════════════
PROFILES
══════════════════════════════════════════════════
Total profiles:     1460
Null bios:          89
All in posts:       False  (should be True)

═══════════════════════════

In [48]:
# This is the oldest relevant post
print(df[df['created_at'] == df['created_at'].min()][['handle', 'text_clean', 'created_at']])

          handle                                         text_clean  \
2892  ladeak.net  Maybe we should use AI then for better underst...   

                     created_at  
2892  2025-06-27 06:01:08+00:00  


In [49]:
# Check which authors do not have saved posts
missing = df_profiles[~df_profiles['handle'].isin(df['handle'])]['handle'].tolist()
print(f"Profiles not in posts: {missing}")

Profiles not in posts: ['pauseai.bsky.social', 'alltheyud-mir-rt.selfhosted.social', 'daveshapi-rt-mirro.selfhosted.social', 'ancadweb.com', 'gandersocial.ca', 'garrytan-mir-rt.selfhosted.social', 'falllover.bsky.social', 'samuelxl5.bsky.social', 'aisafetymem-mir-rt.selfhosted.social', 'bbdarkroast.bsky.social', 'pauseaius.bsky.social', 'rolandbates.bsky.social', 'snuggleproxy.bsky.social', 'gbaspgamer.bsky.social', 'naarna.eurosky.social', 'nowbreezing.ntw.app', 'thisisdonebyhumans.bsky.social', 'nyna-la-reveuse.bsky.social', 'osde8info.bsky.social', 'blockdasher91.bsky.social', 'michelpimpant.eurosky.social', 'pennybee.bsky.social', 'fasenueva.bsky.social', 'aisafetymemes-mirr.selfhosted.social', 'matricejacobine.bsky.social', 'bsky.realhackhistory.org', 'loneicewolf.bsky.social', 'moskov.goodventures.org', 'keytryer.net', 'bollabai.bsky.social', 'turtle428.bsky.social', 'mikehindle.uk', 'nokaccino.bsky.social', 'slowdriver25.bsky.social', 'artificialbodies.net', 'xndxn.bsky.social',

In [51]:
# Delete authors without posts (most likely the post was irrelevant and was filtered out)
df_profiles = df_profiles[df_profiles['handle'].isin(df['handle'])].reset_index(drop=True)
print(f"Profiles after cleanup: {len(df_profiles)}")

# re-save
df_profiles.to_csv('/content/drive/MyDrive/bluesky-data/author_profiles.csv', index=False)

Profiles after cleanup: 1009


In [52]:
print(f"All in posts: {df_profiles['handle'].isin(df['handle']).all()}  (should be True)")
print(f"Null bios: {df_profiles['bio'].isnull().sum()}")

All in posts: True  (should be True)
Null bios: 48


In [54]:
df_profiles.to_csv('/content/drive/MyDrive/bluesky-data/author_profiles.csv', index=False)
print(f"Saved author_profiles.csv - {len(df_profiles)} profiles")
df.to_csv('/content/drive/MyDrive/bluesky-data/posts_clean.csv', index=False)
print(f"Saved posts_clean.csv — {len(df)} posts")

Saved author_profiles.csv - 1009 profiles
Saved posts_clean.csv — 3767 posts


## Analysis

In [ ]:
# RUN FROM HERE ON SAVED DATA
df = pd.read_csv('data/posts_clean.csv')
df_profiles = pd.read_csv('data/author_profiles.csv')
df_graph = pd.read_csv('data/graph.csv')

print(f"Posts loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nGraph DataFrame shape:", df_graph.shape)
print(f"Graph DataFrame columns:", df_graph.columns)
print(f"\nAuthor profile DataFrame shape:", df_profiles.shape)
print(f"Author profile DataFrame columns:", df_profiles.columns)

Posts loaded: 858
Columns: ['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes', 'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply', 'reply_to', 'query', 'text_clean', 'bio', 'user_type']

Graph DataFrame shape: (10662, 3)
Graph DataFrame columns: Index(['source', 'target', 'type'], dtype='str')

Author profile DataFrame shape: (106, 6)
Author profile DataFrame columns: Index(['handle', 'display_name', 'bio', 'followers', 'following',
       'posts_count'],
      dtype='str')
